# Decision Tree

In [1]:
import os
os.chdir("E:\Data Science\ML\Project")

In [2]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, roc_auc_score)

## load the dataset

In [3]:
df = pd.read_csv("data\ChurnGuard_processed.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9259 entries, 0 to 9258
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                9259 non-null   int64  
 1   gender             9259 non-null   object 
 2   tenure_months      9259 non-null   int64  
 3   contract_type      9259 non-null   object 
 4   monthly_charges    9259 non-null   float64
 5   total_charges      9259 non-null   float64
 6   internet_service   9259 non-null   object 
 7   payment_method     9259 non-null   object 
 8   support_calls      9259 non-null   int64  
 9   late_payments      9259 non-null   int64  
 10  online_security    9259 non-null   object 
 11  tech_support       9259 non-null   object 
 12  streaming_service  9259 non-null   object 
 13  senior_citizen     9259 non-null   int64  
 14  family_members     9259 non-null   int64  
 15  churn              9259 non-null   int64  
dtypes: float64(2), int64(7),

In [5]:
X = df.drop(columns="churn")
y = df.churn

## Data Preprocessing & Decision Tree Pipeline

In [6]:
num_var = X.select_dtypes(exclude="object").columns
cat_var = X.select_dtypes(include="object").columns

cat_tran = Pipeline([("encode", OneHotEncoder(drop="first", handle_unknown="ignore"))])
preprocessing = ColumnTransformer([("num", "passthrough", num_var), ("cat", cat_tran, cat_var)])
model = Pipeline([("preprocessing", preprocessing), ("classifier", DecisionTreeClassifier(random_state=38))])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=47)

## Define GridSearch parameters

In [7]:
dt_params = {
    "classifier__criterion": ["gini", "entropy", "log_loss"],
    "classifier__max_depth": [3, 5, 7, 10, 15, None],
    "classifier__min_samples_split": [2, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 5, 10],
    "classifier__max_features": [None, "sqrt", "log2"]
}

## GridSearchCV

In [8]:
grid_dt = GridSearchCV(estimator=model, param_grid=dt_params, scoring="accuracy", cv=5, n_jobs=-1, verbose=1)
grid_dt.fit(X_train, y_train)

Fitting 5 folds for each of 864 candidates, totalling 4320 fits


,estimator,Pipeline(step...m_state=38))])
,param_grid,"{'classifier__criterion': ['gini', 'entropy', ...], 'classifier__max_depth': [3, 5, ...], 'classifier__max_features': [None, 'sqrt', ...], 'classifier__min_samples_leaf': [1, 2, ...], ...}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


## Best parameters

In [9]:
print("Best Parameters:")
print(grid_dt.best_params_)

print("\nBest CV Accuracy:")
print(grid_dt.best_score_)

Best Parameters:
{'classifier__criterion': 'entropy', 'classifier__max_depth': 5, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 5, 'classifier__min_samples_split': 2}

Best CV Accuracy:
0.7754826087709275


## Model Performance Evaluation¶

In [10]:
best_dt = grid_dt.best_estimator_

In [11]:
y_pred = best_dt.predict(X_test)
y_pred_prob = best_dt.predict_proba(X_test)[:, 1]

In [12]:
confusion_matrix(y_test, y_pred)

array([[1358,   20],
       [ 460,   14]])

In [13]:
accuracy_score(y_test, y_pred)

0.7408207343412527

In [14]:
roc_auc_score(y_test, y_pred_prob)

0.5840827530880075

In [15]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.99      0.85      1378
           1       0.41      0.03      0.06       474

    accuracy                           0.74      1852
   macro avg       0.58      0.51      0.45      1852
weighted avg       0.66      0.74      0.65      1852

